# pulse-ep — interactive Jupyter walkthrough

End-to-end demo of consuming the **pulse-ep** REST API from a notebook.

1. Configure the connection from environment variables
2. Log in and list studies / maps
3. Fetch a mesh and visualise it interactively with PyVista
4. Plot the scalar distribution with matplotlib
5. (Bonus) Fit a Gaussian decay with `scipy.optimize.curve_fit`

Make sure the server is running (`pulse-ep-server` or `docker compose --profile server up`)
and that `PULSE_EP_BASE_URL`, `PULSE_EP_USERNAME`, `PULSE_EP_PASSWORD` are set.

## 1. Connection setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import requests

BASE_URL = os.environ.get('PULSE_EP_BASE_URL', 'http://127.0.0.1:5000')
USERNAME = os.environ.get('PULSE_EP_USERNAME', 'admin')
PASSWORD = os.environ.get('PULSE_EP_PASSWORD')
if not PASSWORD:
    raise RuntimeError('PULSE_EP_PASSWORD env var is not set.')

pv.set_jupyter_backend('trame')  # interactive in JupyterLab; use 'static' on headless servers
BASE_URL

## 2. Authenticate

In [ ]:
def pe_login(base_url: str, username: str, password: str) -> str:
    r = requests.post(
        f'{base_url}/login_user',
        json={'username': username, 'password': password},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()['access_token']


def pe_headers(token: str) -> dict[str, str]:
    return {'Authorization': f'Bearer {token}'}


token = pe_login(BASE_URL, USERNAME, PASSWORD)
print('Got JWT (truncated):', token[:24] + '…')

## 3. Browse studies and maps

In [ ]:
studies = pd.DataFrame(
    requests.get(f'{BASE_URL}/list_studies', headers=pe_headers(token), timeout=15).json()
)
studies.head(10)

In [ ]:
study_id = int(os.environ.get('PE_STUDY_ID', studies['id'].iloc[0]))
maps = pd.DataFrame(
    requests.get(
        f'{BASE_URL}/list_epmaps_in_study/{study_id}',
        headers=pe_headers(token), timeout=15,
    ).json()
)
maps.head(10)

## 4. Fetch one map's mesh

In [ ]:
map_id = int(os.environ.get('PE_MAP_ID', maps['id'].iloc[0]))
params = {'map_id': map_id, 'scalar_name': 'act', 'distance': 5.0}
payload = requests.get(
    f'{BASE_URL}/get_mesh_data', params=params, headers=pe_headers(token), timeout=60,
).json()

vertices = np.asarray(payload['mesh_data']['vertices'], dtype=float)
faces_raw = np.asarray(payload['mesh_data']['faces'], dtype=np.int64)
scalars = np.asarray(
    [np.nan if v is None else float(v) for v in payload['mesh_data']['scalar_data']],
    dtype=float,
)

# PyVista's PolyData expects a flat connectivity array prefixed with '3' per triangle
faces = np.column_stack([np.full(len(faces_raw), 3, dtype=np.int64), faces_raw]).ravel()
mesh = pv.PolyData(vertices, faces)
mesh['act'] = scalars
f'Loaded mesh: {mesh.n_points} vertices, {mesh.n_cells} faces, scalar non-NaN = {np.isfinite(scalars).sum()}'

## 5. Interactive 3D rendering

In [ ]:
pl = pv.Plotter(notebook=True)
pl.add_mesh(mesh, scalars='act', nan_color='lightgrey', cmap='turbo', smooth_shading=True)
pl.add_text(f'pulse-ep map {map_id}', position='upper_left')
pl.show()

## 6. Scalar distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(scalars[np.isfinite(scalars)], bins=40, color='#3a76ff', edgecolor='white')
ax.set_xlabel('Scalar value (act)')
ax.set_ylabel('Vertex count')
ax.set_title(f'pulse-ep map {map_id} — scalar distribution')
fig.tight_layout()
fig.show()

## 7. (Bonus) Gaussian decay fit

Same fit as the R demo: $s(d) = A\,e^{-d^2 / (2\sigma^2)} + B$, using **Euclidean** distance from the maximum-scalar vertex.

For the proper σ-resolution analysis based on Heat-Method geodesic distances,
see [`pulse-ep-decay`](https://gitlab.willert.net/sw/pulse-ep-decay).

In [ ]:
from scipy.optimize import curve_fit

mask = np.isfinite(scalars)
v = vertices[mask]
s = scalars[mask]
if len(s) >= 20:
    i_max = int(np.argmax(s))
    d = np.linalg.norm(v - v[i_max], axis=1)

    def gauss(d, A, sigma, B):
        return A * np.exp(-(d**2) / (2 * sigma**2)) + B

    p0 = (s.max() - s.min(), (d.max() - d.min()) / 4, s.min())
    popt, _ = curve_fit(gauss, d, s, p0=p0, maxfev=2000)
    A, sigma, B = popt
    print(f'A     = {A:.3f}')
    print(f'sigma = {sigma:.3f} mm')
    print(f'B     = {B:.3f}')

    grid = np.linspace(0, d.max(), 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(d, s, s=4, alpha=0.25)
    ax.plot(grid, gauss(grid, *popt), color='#e34a33', linewidth=2)
    ax.set_xlabel('Euclidean distance from max-scalar vertex [mm]')
    ax.set_ylabel('Scalar value')
    ax.set_title(f'Gaussian decay — σ ≈ {sigma:.2f} mm')
    fig.tight_layout()
    fig.show()
else:
    print('Too few non-NaN scalars for a Gaussian fit.')